# Notebook 01 — Coleta da Câmara dos Deputados

**Sprint 1 — Lei e Política**

Este notebook coleta, a partir da API aberta da Câmara, os dados necessários para as etapas seguintes do pipeline:

| Endpoint | Dado coletado | Destino no banco |
|---|---|---|
| `/deputados` | cadastro dos 513 deputados | `parlamentares` |
| `/proposicoes` | ementas para clustering | `proposicoes` |
| `/votacoes` | votações da legislatura 57 | `votacoes` |
| `/votacoes/{id}/votos` | votos individuais nominais | `votos` |

**Decisões de projeto registradas aqui:**
- Somente votações **nominais** são mantidas (votos individuais retornados pela API); votações simbólicas são descartadas e a proporção é logada.
- `tipoVoto` é normalizado para `{'favoravel', 'contrario', 'abstencao'}` antes de qualquer inserção.
- Todos os JSONs brutos são salvos em `data/raw/` antes de qualquer transformação (coleta cara, não repetir por erro de processamento).
- Upsert idempotente: reexecutar o notebook não duplica registros.

In [1]:
import sys
sys.path.insert(0, '..')

import logging
import time
from collections import defaultdict

import pandas as pd

from src.coleta import (
    paginar_camara,
    _get_com_retry,
    normalizar_voto,
    salvar_raw,
    carregar_raw,
    BASE_CAMARA,
)
from src.db import (
    upsert_parlamentares,
    upsert_proposicoes,
    upsert_votacoes,
    upsert_votos,
    buscar_todos,
    buscar_id_interno,
    buscar_id_votacao,
)

log = logging.getLogger('01_coleta_camara')
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
print('Módulos carregados.')

Módulos carregados.


## 1. Deputados da legislatura 57 (2023–2027)

In [2]:
deputados_raw = paginar_camara('deputados', params={'idLegislatura': 57})
salvar_raw('camara_deputados.json', deputados_raw)
print(f'Total de deputados coletados: {len(deputados_raw)}')

18:37:29 [INFO]   deputados — página 1 → 100 itens acumulados
18:37:29 [INFO]   deputados — página 2 → 200 itens acumulados
18:37:32 [INFO]   deputados — página 3 → 300 itens acumulados
18:37:32 [INFO]   deputados — página 4 → 400 itens acumulados
18:37:34 [INFO]   deputados — página 5 → 500 itens acumulados
18:37:35 [INFO]   deputados — página 6 → 600 itens acumulados
18:37:37 [INFO]   deputados — página 7 → 700 itens acumulados
18:37:38 [INFO]   deputados — página 8 → 800 itens acumulados
18:37:41 [INFO]   deputados — página 9 → 872 itens acumulados
18:37:44 [INFO] Salvo: /home/brandao/Documentos/trabalhoCiênciadeDados/data/raw/camara_deputados.json (872 registros)


Total de deputados coletados: 872


In [3]:
registros_parl = [
    {
        'id_externo': int(d['id']),
        'casa': 'camara',
        'nome': d.get('nome', ''),
        'partido': d.get('siglaPartido', ''),
        'uf': d.get('siglaUf', ''),
        'foto_url': d.get('urlFoto', ''),
    }
    for d in deputados_raw
]

total_inserido = upsert_parlamentares(registros_parl)
print(f'Parlamentares (câmara) inseridos/atualizados: {total_inserido}')

18:37:52 [INFO] HTTP Request: POST https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/parlamentares?on_conflict=id_externo%2Ccasa&columns=%22casa%22%2C%22nome%22%2C%22foto_url%22%2C%22id_externo%22%2C%22uf%22%2C%22partido%22 "HTTP/2 201 Created"
18:37:52 [INFO] upsert parlamentares: 644 registros


Parlamentares (câmara) inseridos/atualizados: 644


## 2. Proposições (corpus para clustering de temas)

Coletamos proposições desde o início da legislatura. O campo `ementa` será o corpus do TF-IDF na Sprint 2.

In [ ]:
from datetime import date, timedelta

def janelas_trimestrais(inicio: str, fim: str):
    """Gera pares (inicio, fim) em janelas de ~90 dias."""
    d = date.fromisoformat(inicio)
    fim_d = date.fromisoformat(fim)
    while d <= fim_d:
        proximo = d.replace(day=1)
        mes = proximo.month + 3
        ano = proximo.year + (mes - 1) // 12
        mes = ((mes - 1) % 12) + 1
        proximo = date(ano, mes, 1) - timedelta(days=1)
        proximo = min(proximo, fim_d)
        yield d.isoformat(), proximo.isoformat()
        d = proximo + timedelta(days=1)

# Apenas tipos com ementa legislativa substantiva.
# Requerimentos, ofícios e indicações não contribuem para o corpus de temas.
TIPOS_LEGISLATIVOS = ['PL', 'PEC', 'PLP', 'MPV', 'PDL']

DATA_INICIO = '2025-01-01'
DATA_FIM    = '2026-06-12'

proposicoes_raw = []
for sigla in TIPOS_LEGISLATIVOS:
    for ini, fim_jan in janelas_trimestrais(DATA_INICIO, DATA_FIM):
        dados = paginar_camara(
            'proposicoes',
            params={
                'siglaTipo': sigla,
                'dataApresentacaoInicio': ini,
                'dataApresentacaoFim': fim_jan,
                'ordenarPor': 'id',
                'ordem': 'ASC',
            },
        )
        proposicoes_raw.extend(dados)
    log.info('Tipo %s concluído: acumulado %d proposições', sigla, len(proposicoes_raw))

salvar_raw('camara_proposicoes.json', proposicoes_raw)
print(f'Total de proposições coletadas: {len(proposicoes_raw)}')

In [5]:
import re

def data_segura(s):
    """Extrai YYYY-MM-DD de uma string ISO, retorna None se inválido."""
    if not s:
        return None
    m = re.match(r'(\d{4}-\d{2}-\d{2})', str(s))
    return m.group(1) if m else None

registros_prop = []
for p in proposicoes_raw:
    ementa = p.get('ementa', '') or ''
    if len(ementa.strip()) < 10:   # ignorar ementas vazias
        continue
    registros_prop.append({
        'id_externo': int(p['id']),
        'casa': 'camara',
        'ementa': ementa.strip(),
        'keywords': p.get('keywords', '') or '',
        'data': data_segura(p.get('dataApresentacao')),
    })

print(f'Proposições com ementa válida: {len(registros_prop)} / {len(proposicoes_raw)}')
total_prop = upsert_proposicoes(registros_prop)
print(f'Proposições inseridas/atualizadas: {total_prop}')

Proposições com ementa válida: 0 / 0
Proposições inseridas/atualizadas: 0


## 3. Votações

Coletamos os metadados de cada votação. Na próxima etapa filtramos apenas as que possuem votos nominais.

In [ ]:
votacoes_raw = []
for ini, fim_jan in janelas_trimestrais(DATA_INICIO, DATA_FIM):
    dados = paginar_camara(
        'votacoes',
        params={
            'dataInicio': ini,
            'dataFim': fim_jan,
            'ordenarPor': 'data',
            'ordem': 'ASC',
        },
    )
    votacoes_raw.extend(dados)
    log.info('Votações %s → %s: %d registros (total: %d)', ini, fim_jan, len(dados), len(votacoes_raw))

salvar_raw('camara_votacoes.json', votacoes_raw)
print(f'Total de votações coletadas: {len(votacoes_raw)}')

## 4. Votos nominais — coleta com filtro

**Decisão metodológica:** a maioria das votações da Câmara é simbólica (voz/aclamação) e não produz registro nominal por deputado. Essas votações são inúteis para o modelo supervisionado e são descartadas. Logamos a proporção para transparência.

In [ ]:
# Busca o mapa id_externo → id interno dos parlamentares da câmara
parl_db = buscar_todos('parlamentares', 'id,id_externo,casa')
mapa_parl = {(r['id_externo'], r['casa']): r['id'] for r in parl_db}
print(f'Parlamentares no banco: {len(parl_db)}')

In [ ]:
# Busca o mapa id_externo → proposicao_id no banco
prop_db = buscar_todos('proposicoes', 'id,id_externo,casa')
mapa_prop = {(r['id_externo'], r['casa']): r['id'] for r in prop_db}
print(f'Proposições no banco: {len(prop_db)}')

In [ ]:
total_votacoes = len(votacoes_raw)
votacoes_nominais = 0
votacoes_sem_proposicao = 0
votos_nao_mapeados_tipo = defaultdict(int)
votos_parlamentar_ausente = 0

registros_votacoes = []
registros_votos_todos = []

for idx, v in enumerate(votacoes_raw):
    vid = v.get('id', '')
    if not vid:
        continue

    # Buscar votos desta votação
    url_votos = f'{BASE_CAMARA}/votacoes/{vid}/votos'
    resp = _get_com_retry(url_votos)
    if resp.status_code != 200:
        continue

    votos_lista = resp.json().get('dados', [])
    time.sleep(0.3)

    # Filtrar votações simbólicas
    if not votos_lista:
        continue

    votacoes_nominais += 1

    # Relacionar à proposição
    prop_obj = v.get('proposicaoObjeto') or {}
    prop_id_externo = prop_obj.get('id') if isinstance(prop_obj, dict) else None
    proposicao_id = mapa_prop.get((int(prop_id_externo), 'camara')) if prop_id_externo else None

    if proposicao_id is None:
        votacoes_sem_proposicao += 1
        # Mantemos a votação mas sem link de proposição

    # Extrair data
    data_vot = data_segura(v.get('dataHoraInicio') or v.get('dataHoraRegistro', ''))

    registros_votacoes.append({
        'id_externo': str(vid),
        'proposicao_id': proposicao_id,
        'data': data_vot,
    })

    for voto in votos_lista:
        tipo_raw = voto.get('tipoVoto', '')
        tipo_norm = normalizar_voto(tipo_raw)

        if tipo_norm is None:
            votos_nao_mapeados_tipo[tipo_raw] += 1
            continue

        dep = voto.get('deputado_', {}) or {}
        dep_id_ext = dep.get('id')
        if not dep_id_ext:
            votos_parlamentar_ausente += 1
            continue

        parl_id = mapa_parl.get((int(dep_id_ext), 'camara'))
        if parl_id is None:
            votos_parlamentar_ausente += 1
            continue

        registros_votos_todos.append({
            '_votacao_id_externo': str(vid),
            '_parlamentar_id': parl_id,
            'voto': tipo_norm,
        })

    if (idx + 1) % 50 == 0:
        log.info('Votações processadas: %d/%d | Nominais até agora: %d', idx + 1, total_votacoes, votacoes_nominais)

salvar_raw('camara_votos_nominais.json', registros_votos_todos)

print(f'\n=== Resumo da coleta de votos ===')
print(f'Total de votações na API:         {total_votacoes}')
print(f'Votações nominais (com votos):    {votacoes_nominais} ({100*votacoes_nominais/total_votacoes:.1f}%)')
print(f'Descartadas (simbólicas):         {total_votacoes - votacoes_nominais} ({100*(total_votacoes - votacoes_nominais)/total_votacoes:.1f}%)')
print(f'Votações sem proposição linkada:  {votacoes_sem_proposicao}')
print(f'Votos com tipoVoto desconhecido:  {dict(votos_nao_mapeados_tipo)}')
print(f'Votos sem parlamentar no banco:   {votos_parlamentar_ausente}')
print(f'Total de votos nominais coletados:{len(registros_votos_todos)}')

## 5. Inserção no Supabase

Primeiro inserimos as votações para obter os IDs internos. Depois usamos esses IDs para inserir os votos.

In [ ]:
upsert_votacoes(registros_votacoes)
print('Votações inseridas.')

In [ ]:
# Recarregar mapa de votações após inserção
vot_db = buscar_todos('votacoes', 'id,id_externo')
mapa_vot = {r['id_externo']: r['id'] for r in vot_db}
print(f'Votações no banco: {len(mapa_vot)}')

# Substituir id_externo pelo id interno
votos_para_inserir = []
votos_sem_votacao = 0
for vr in registros_votos_todos:
    vid_interno = mapa_vot.get(vr['_votacao_id_externo'])
    if vid_interno is None:
        votos_sem_votacao += 1
        continue
    votos_para_inserir.append({
        'votacao_id': vid_interno,
        'parlamentar_id': vr['_parlamentar_id'],
        'voto': vr['voto'],
    })

print(f'Votos prontos para inserção: {len(votos_para_inserir)}')
if votos_sem_votacao:
    print(f'AVISO: {votos_sem_votacao} votos descartados por votação não encontrada no banco')

In [ ]:
upsert_votos(votos_para_inserir)
print('Votos inseridos.')

## 6. Célula de sanidade — contagens finais

Verificação final: confirmar que os dados estão no banco com as contagens esperadas.

In [ ]:
from src.db import get_client

client = get_client()

def contar(tabela, filtros=None):
    q = client.table(tabela).select('id', count='exact')
    if filtros:
        for col, val in filtros.items():
            q = q.eq(col, val)
    r = q.execute()
    return r.count

print('=== Sanidade do banco — após Sprint 1 (Câmara) ===')
print(f'parlamentares (câmara):  {contar("parlamentares", {"casa": "camara"})}')
print(f'proposições (câmara):    {contar("proposicoes", {"casa": "camara"})}')
print(f'votações:                {contar("votacoes")}')
print(f'votos:                   {contar("votos")}')

total_vot = contar('votacoes')
pct_desc = 100 * (total_votacoes - votacoes_nominais) / total_votacoes if total_votacoes else 0
print(f'\n% votações simbólicas descartadas: {pct_desc:.1f}%')
print('\nDistribuição dos votos:')
for tipo in ['favoravel', 'contrario', 'abstencao']:
    n = contar('votos', {'voto': tipo})
    print(f'  {tipo}: {n}')